# Read the val dataset

In [1]:
import pandas as pd 
import numpy as np 

df = pd.read_csv('../data/dontpatronizeme_pcl.tsv', sep='\t')

# Remove rows with NA labels 
df = df.dropna() 

# Add a bool_labels column for binary classification
df["bool_labels"] = df["label"] > 1   # is PCL if >1
val_labels = pd.read_csv('../data/dev_semeval_parids-labels.csv')["par_id"]  # extract the labels for val dataset 
df_val = df[df["par_id"].isin(val_labels)].reset_index() 

# Read the prediction results: 
- baseline.txt (Roberta) 
- only_oversample.txt (Baseline Roberta + Oversample)
- roberta_ensemble.txt (Baseline Roberta + Oversample + Context)
- oversample_context_cr.txt (Baseline Roberta + Oversample + Context + Coreference resolution)
- bert_ensemble.txt (BERT + Oversample + Context)
- final.txt (Same as dev.txt, ensembles) 

In [2]:
with open("baseline.txt", "r") as f: 
    y_pred_baseline = np.array([int(x) for x in f.read().split("\n")]) 

with open("only_oversample.txt", "r") as f: 
    y_pred_only_oversample = np.array([int(x) for x in f.read().split("\n")]) 

with open("roberta_ensemble.txt", "r") as f: 
    y_pred_roberta_ensemble = np.array([int(x) for x in f.read().split("\n")]) 

with open("oversample_context_cr.txt", "r") as f: 
    y_pred_oversample_context_cr = np.array([int(x) for x in f.read().split("\n")]) 

with open("bert_ensemble.txt", "r") as f: 
    y_pred_bert_ensemble = np.array([int(x) for x in f.read().split("\n")]) 

with open("final.txt", "r") as f: 
    y_pred_final = np.array([int(x) for x in f.read().split("\n")]) 

y_true = df_val["bool_labels"]

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# assert that the f1 values remain the same 
print(f1_score(y_true, y_pred_baseline))
print(f1_score(y_true, y_pred_only_oversample))
print(f1_score(y_true, y_pred_roberta_ensemble))
print(f1_score(y_true, y_pred_oversample_context_cr))
print(f1_score(y_true, y_pred_bert_ensemble))
print(f1_score(y_true, y_pred_final))

0.5772151898734177
0.5871121718377088
0.6193548387096774
0.5653206650831354
0.5511811023622047
0.6299559471365639


# Examples of tp, tn, fp, fn for each class

In [4]:
pd.set_option("display.max_colwidth", None) 

keywords = ['homeless', 'disabled', 'poor-families', 'in-need', 'women', 'refugee', 'migrant', 'hopeless', 'immigrant', 'vulnerable']

# Extract examples of TP, TN, FP, FN, for each keyword
for keyword in keywords: 
    print("="*50)
    print(f"Keyword: {keyword}")
    print("="*50)
    tp = df_val[(y_true == 1) & (y_pred_final == 1)][["text"]].head()
    tn = df_val[(y_true == 0) & (y_pred_final == 0)][["text"]].head()
    fp = df_val[(y_true == 0) & (y_pred_final == 1)][["text"]].head()
    fn = df_val[(y_true == 1) & (y_pred_final == 0)][["text"]].head()

    print(tp) 
    print(tn)
    print(fp) 
    print(fn)

Keyword: homeless
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        text
3                                                                                                           When some people feel causing problem for some others by breaking into their homes to steal is n't too good , they just result to begging . You now see people without deformities begging , when some people who are disabled work to feed their mouth . You then ask , what

# Metrics for each keyword

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

for keyword in keywords: 
    print("="*50)
    print(f"Keyword: {keyword}")
    print("="*50)

    mask = df_val["keyword"] == keyword 

    print(f"Accuracy: {accuracy_score(y_true[mask], y_pred_final[mask]):.3f}")
    print(f"Precision: {precision_score(y_true[mask], y_pred_final[mask]):.3f}")
    print(f"Recall: {recall_score(y_true[mask], y_pred_final[mask]):.3f}")
    print(f"F1: {f1_score(y_true[mask], y_pred_final[mask]):.3f}")

print("="*50)
print(f"Overall")
print("="*50)

print(f"Accuracy: {accuracy_score(y_true, y_pred_final):.3f}")
print(f"Precision: {precision_score(y_true, y_pred_final):.3f}") 
print(f"Recall: {recall_score(y_true, y_pred_final):.3f}")  
print(f"F1: {f1_score(y_true, y_pred_final):.3f}")

Keyword: homeless
Accuracy: 0.910
Precision: 0.647
Recall: 0.759
F1: 0.698
Keyword: disabled
Accuracy: 0.912
Precision: 0.421
Recall: 0.571
F1: 0.485
Keyword: poor-families
Accuracy: 0.821
Precision: 0.545
Recall: 0.632
F1: 0.585
Keyword: in-need
Accuracy: 0.920
Precision: 0.653
Recall: 0.970
F1: 0.780
Keyword: women
Accuracy: 0.927
Precision: 0.400
Recall: 0.429
F1: 0.414
Keyword: refugee
Accuracy: 0.936
Precision: 0.526
Recall: 0.769
F1: 0.625
Keyword: migrant
Accuracy: 0.971
Precision: 0.400
Recall: 0.400
F1: 0.400
Keyword: hopeless
Accuracy: 0.889
Precision: 0.528
Recall: 0.731
F1: 0.613
Keyword: immigrant
Accuracy: 0.986
Precision: 0.833
Recall: 0.714
F1: 0.769
Keyword: vulnerable
Accuracy: 0.914
Precision: 0.536
Recall: 0.750
F1: 0.625
Overall
Accuracy: 0.920
Precision: 0.561
Recall: 0.719
F1: 0.630
